# No-vig fair odds and expected value (EV)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JacobiusMakes/parlayapi-notebooks/blob/main/02-no-vig-and-ev.ipynb)

A sportsbook's prices always sum to more than 100% implied probability. The extra
slice is the vig (also called the margin or juice). This notebook:

1. strips the vig **two ways** (proportional and additive) to recover fair odds,
2. verifies the math against the canonical cases used by ParlayAPI's own
   [no-vig calculator](https://parlay-api.com/tools/no-vig-calculator)
   (for example -110 / -110 devigs to +100 / +100 with 4.76% vig),
3. computes the **EV of any price against the no-vig fair line**, the same
   convention as the [EV calculator](https://parlay-api.com/tools/ev-calculator),
4. runs it on live odds.

Works with no key (demo endpoint). A [free key](https://parlay-api.com/signup) gets every book and market.

In [1]:
# ---- Config: paste your API key between the quotes ----
# Get a free key at https://parlay-api.com/signup (free tier, no card required).
# Leave it empty to run against the keyless demo endpoint instead.
API_KEY = ""

import os
API_KEY = (API_KEY or os.environ.get("PARLAYAPI_KEY", "")).strip()
BASE_URL = "https://parlay-api.com"
SPORT = "baseball_mlb"  # also try: basketball_nba, americanfootball_nfl, icehockey_nhl, soccer_epl

if API_KEY:
    print("API key set: using the full keyed endpoints.")
else:
    print("No API key set: falling back to the keyless demo endpoint.")
    print("The demo serves the first 5 events per sport, moneyline (h2h) only,")
    print("capped at 60 requests per hour. Paste a free key above for all events,")
    print("all 30+ books, and every market:", "https://parlay-api.com/signup")

No API key set: falling back to the keyless demo endpoint.
The demo serves the first 5 events per sport, moneyline (h2h) only,
capped at 60 requests per hour. Paste a free key above for all events,
all 30+ books, and every market: https://parlay-api.com/signup


In [2]:
# American <-> decimal conversions, matching the conventions used by
# https://parlay-api.com/tools/no-vig-calculator and /tools/parlay-calculator.

def american_to_decimal(a):
    """+150 -> 2.5, -110 -> 1.9091. Valid American odds are >= +100 or <= -100."""
    a = float(a)
    if abs(a) < 100:
        raise ValueError(f"{a} is not a valid American price (must be >= +100 or <= -100)")
    if a > 0:
        return 1 + a / 100
    return 1 + 100 / (-a)

def decimal_to_american(d):
    """2.5 -> +150, 1.9091 -> -110 (rounded to the nearest integer)."""
    d = float(d)
    if d <= 1:
        raise ValueError(f"decimal odds must be > 1, got {d}")
    if d >= 2:
        return round((d - 1) * 100)
    return round(-100 / (d - 1))

def implied_prob(decimal_odds):
    """Implied win probability of decimal odds (includes the vig)."""
    return 1.0 / float(decimal_odds)

def fmt_american(a):
    return ("+" if a > 0 else "") + str(int(a))

## Way 1: proportional (multiplicative) devig

Convert every price to an implied probability, sum them to get the **overround**
(a number just above 1), then divide each probability by the overround. This is the
transparent, assumption-free default, and it is the method ParlayAPI's calculator
and EV tooling use.

The **vig** reported here is `overround - 1`. At -110 / -110 each side implies
52.38%, the overround is 1.0476, and the vig is 4.76%. (Some books instead quote
`(overround - 1) / overround`, which gives 4.55% for the same market. Same market,
different convention; know which one you are reading.)

In [3]:
def devig_proportional(american_prices):
    """Proportional (multiplicative) devig of one market.

    Returns dict with raw implied probs, overround, vig, fair probs,
    fair decimal odds, and fair American odds.
    """
    decs = [american_to_decimal(a) for a in american_prices]
    raw = [1.0 / d for d in decs]
    overround = sum(raw)
    fair = [r / overround for r in raw]
    fair_dec = [1.0 / p for p in fair]
    return {
        "raw_implied": raw,
        "overround": overround,
        "vig": overround - 1,
        "fair_probs": fair,
        "fair_decimal": fair_dec,
        "fair_american": [decimal_to_american(d) for d in fair_dec],
    }

out = devig_proportional([-110, -110])
print(f"-110 / -110  ->  fair probs {out['fair_probs']}, "
      f"fair American {out['fair_american']}, vig {out['vig']*100:.2f}%")

-110 / -110  ->  fair probs [0.5, 0.5], fair American [100, 100], vig 4.76%


## Way 2: additive devig

Subtract an equal share of the margin from every outcome's implied probability:
`fair_i = raw_i - (overround - 1) / n`.

Compared with proportional devig, the additive method takes the same absolute
margin off each outcome, which shifts relatively more of the vig onto longshots'
prices. On heavy longshots it can even push a probability to zero or below, so it
needs a guard. It is worth knowing because the two methods bracket what most
books actually do; power and Shin devigs (not implemented here) sit in the same
family but need extra assumptions.

In [4]:
def devig_additive(american_prices):
    """Additive devig: remove an equal share of the margin from each outcome."""
    decs = [american_to_decimal(a) for a in american_prices]
    raw = [1.0 / d for d in decs]
    overround = sum(raw)
    share = (overround - 1) / len(raw)
    fair = [r - share for r in raw]
    if min(fair) <= 0:
        raise ValueError("additive devig produced a probability <= 0; "
                         "use proportional for markets with heavy longshots")
    fair_dec = [1.0 / p for p in fair]
    return {
        "raw_implied": raw,
        "overround": overround,
        "vig": overround - 1,
        "fair_probs": fair,
        "fair_decimal": fair_dec,
        "fair_american": [decimal_to_american(d) for d in fair_dec],
    }

# Side by side on a lopsided two-way market:
prices = [-200, +170]
p = devig_proportional(prices)
a = devig_additive(prices)
print(f"market {prices}, vig {p['vig']*100:.2f}%")
print(f"  proportional fair: {[round(x, 4) for x in p['fair_probs']]} "
      f"-> {[fmt_american(x) for x in p['fair_american']]}")
print(f"  additive fair:     {[round(x, 4) for x in a['fair_probs']]} "
      f"-> {[fmt_american(x) for x in a['fair_american']]}")

market [-200, 170], vig 3.70%
  proportional fair: [0.6429, 0.3571] -> ['-180', '+180']
  additive fair:     [0.6481, 0.3519] -> ['-184', '+184']


## Verify against the site calculator's canonical cases

These are the exact test vectors the [no-vig calculator](https://parlay-api.com/tools/no-vig-calculator)
self-tests on load. If any assert fires, the math above has drifted.

In [5]:
CANONICAL = [
    # (american prices, vig, fair probs, fair american)
    ([-110, -110],      0.0476, [0.5, 0.5],               [100, 100]),
    ([-120, +100],      0.0455, [0.5217, 0.4783],         [-109, 109]),
    ([+150, -170],      0.0296, [0.3885, 0.6115],         [157, -157]),
    ([+150, +220, +240],0.0066, [0.3974, 0.3104, 0.2922], [152, 222, 242]),
    ([-110, +100],      0.0238, [0.5116, 0.4884],         [-105, 105]),
    ([-200, +170],      0.0370, [0.6429, 0.3571],         [-180, 180]),
]

for prices, vig, fair, fair_am in CANONICAL:
    got = devig_proportional(prices)
    assert abs(got["vig"] - vig) < 5e-5, (prices, got["vig"])
    for g, want in zip(got["fair_probs"], fair):
        assert abs(g - want) < 5e-5, (prices, got["fair_probs"])
    assert got["fair_american"] == fair_am, (prices, got["fair_american"])

print(f"all {len(CANONICAL)} canonical devig cases pass")
assert devig_proportional([-110, -110])["fair_american"] == [100, 100]
print("-110/-110 -> +100/+100 with vig "
      f"{devig_proportional([-110, -110])['vig']*100:.2f}% (the textbook case)")

all 6 canonical devig cases pass
-110/-110 -> +100/+100 with vig 4.76% (the textbook case)


## EV of a price against the no-vig fair line

Once you have a fair win probability `p`, the expected value of a price per $1
staked is:

```
EV per $1 = p * (dec - 1) - (1 - p)
```

where `dec` is the decimal odds of the price you can actually bet. Multiply by 100
for EV percent, or by your stake for dollars. Positive means the bet wins more, on
average, than it risks. This is the exact formula behind the
[EV calculator](https://parlay-api.com/tools/ev-calculator).

In [6]:
def ev_per_dollar(fair_prob, american_price):
    dec = american_to_decimal(american_price)
    return fair_prob * (dec - 1) - (1 - fair_prob)

# Example: the market says -110/-110 (fair 50%), but one book hangs +105 on a side.
fair = devig_proportional([-110, -110])["fair_probs"][0]
ev = ev_per_dollar(fair, +105)
print(f"fair p = {fair:.4f}, price +105  ->  EV {ev*100:+.2f}% of stake")
assert abs(ev - 0.025) < 1e-9  # 0.5 * 1.05 - 0.5

fair p = 0.5000, price +105  ->  EV +2.50% of stake


## Run it on live odds

For each event we devig every book's moneyline separately, average the fair
probabilities across books into a simple consensus, then score every available
price against that consensus. Rows at the top are the closest to +EV right now.

Honest caveats: without a key the demo feed carries only a few books, so the
consensus is thin; a real workflow devigs a sharp book (or many books) and knows
that a fair price is the market's opinion, not the truth.

In [7]:
import requests

def fetch_odds(sport=None, markets="h2h,spreads,totals", odds_format="american"):
    """Fetch current odds as a list of event dicts.

    Keyed:   GET /v1/sports/{sport}/odds returns a bare JSON array of events
             (the-odds-api compatible shape).
    Keyless: GET /v1/try/{sport}/odds returns a demo envelope instead: the
             events are nested under the "events" key, next to demo metadata
             like demo_message and demo_remaining_hour. The two shapes are
             NOT the same at the top level, so we unwrap here.

    Each event: id, home_team, away_team, commence_time, and
    bookmakers[] -> markets[] -> outcomes[] with American prices by default.
    """
    sport = sport or SPORT
    if API_KEY:
        resp = requests.get(
            f"{BASE_URL}/v1/sports/{sport}/odds",
            params={"markets": markets, "oddsFormat": odds_format},
            headers={"X-API-Key": API_KEY},
            timeout=30,
        )
        resp.raise_for_status()
        return resp.json()
    resp = requests.get(f"{BASE_URL}/v1/try/{sport}/odds", timeout=30)
    resp.raise_for_status()
    payload = resp.json()
    # Demo envelope: {"demo": true, "demo_message": "...", "events": [...]}
    return payload.get("events", [])

In [8]:
import pandas as pd

try:
    events = fetch_odds()
except Exception as exc:
    events = []
    print(f"Fetch failed ({exc}). Check your connection or key and re-run this cell.")
rows = []
for ev in events:
    label = f"{ev['away_team']} at {ev['home_team']}"
    for bm in ev.get("bookmakers", []):
        for mkt in bm.get("markets", []):
            if mkt["key"] != "h2h":
                continue
            outs = mkt.get("outcomes", [])
            if len(outs) < 2:
                continue
            try:
                devig = devig_proportional([o["price"] for o in outs])
            except ValueError:
                continue
            for o, fair_p in zip(outs, devig["fair_probs"]):
                rows.append({"event": label, "outcome": o["name"], "bookmaker": bm["key"],
                             "price": o["price"], "fair_prob_this_book": fair_p})

odds = pd.DataFrame(rows)
if odds.empty:
    print("No h2h markets fetched right now. Re-run later or change SPORT.")
else:
    consensus = (odds.groupby(["event", "outcome"])["fair_prob_this_book"]
                     .mean().rename("consensus_fair_prob").reset_index())
    odds = odds.merge(consensus, on=["event", "outcome"])
    odds["ev_pct"] = [
        100 * ev_per_dollar(p, price)
        for p, price in zip(odds["consensus_fair_prob"], odds["price"])
    ]
    n_books = odds["bookmaker"].nunique()
    print(f"{len(odds)} prices from {n_books} books, scored against a {n_books}-book consensus")
    display(odds.sort_values("ev_pct", ascending=False).head(10).round(4))

130 prices from 13 books, scored against a 13-book consensus


,event,outcome,bookmaker,price,fair_prob_this_book,consensus_fair_prob,ev_pct
30,Cincinnati Reds at Chicago Cubs,Chicago Cubs,fliff,-140,0.5506,0.6259,7.3044
61,Kansas City Royals at Cleveland Guardians,Kansas City Royals,novig,141,0.4128,0.4204,1.3158
49,Cincinnati Reds at Chicago Cubs,Cincinnati Reds,caesars,170,0.3611,0.3741,0.9956
45,Cincinnati Reds at Chicago Cubs,Cincinnati Reds,bet365,170,0.3633,0.3741,0.9956
41,Cincinnati Reds at Chicago Cubs,Cincinnati Reds,prophetx,170,0.3684,0.3741,0.9956
37,Cincinnati Reds at Chicago Cubs,Cincinnati Reds,novig,170,0.3684,0.3741,0.9956
13,Colorado Rockies at Atlanta Braves,Colorado Rockies,draftkings,206,0.3213,0.3297,0.8993
43,Cincinnati Reds at Chicago Cubs,Cincinnati Reds,draftkings,169,0.3655,0.3741,0.6216
92,Los Angeles Dodgers at Detroit Tigers,Detroit Tigers,bet365,170,0.3633,0.3723,0.5187
113,Texas Rangers at Milwaukee Brewers,Texas Rangers,prophetx,150,0.3975,0.4007,0.1750


## Cross-check with the hosted EV scanner

ParlayAPI also runs this scan server-side across its whole book list, anchored on a
sharp book instead of a thin consensus. The keyless demo version is
`GET /v1/try/{sport}/ev` (same demo envelope pattern: results are nested inside
demo metadata). With a key, use `GET /v1/sports/{sport}/ev`.

In [9]:
try:
    if API_KEY:
        r = requests.get(f"{BASE_URL}/v1/sports/{SPORT}/ev",
                         headers={"X-API-Key": API_KEY}, timeout=30)
    else:
        r = requests.get(f"{BASE_URL}/v1/try/{SPORT}/ev", timeout=30)
except requests.RequestException as exc:
    r = None
    print(f"EV endpoint unreachable right now ({exc}); skipping the cross-check.")
if r is not None and r.ok:
    payload = r.json()
    opps = payload.get("opportunities", []) if isinstance(payload, dict) else payload
    if isinstance(payload, dict) and payload.get("method"):
        print("method:", payload["method"])
    for o in opps[:5]:
        print(f"  {o.get('side')} ({o.get('away_team')} at {o.get('home_team')}) "
              f"{fmt_american(o.get('price', 0))} at {o.get('book')}: "
              f"edge {o.get('edge_pct')}% vs {o.get('sharp_anchor')} fair")
    if not opps:
        print("No +EV opportunities surfaced right now.")
elif r is not None:
    print(f"EV endpoint returned HTTP {r.status_code}; see " + BASE_URL + "/docs for details.")

method: devig_pinnacle_h2h_vs_book
  Chicago Cubs (Cincinnati Reds at Chicago Cubs) -140 at Fliff: edge 4.408% vs pinnacle fair
  Colorado Rockies (Colorado Rockies at Atlanta Braves) +206 at DraftKings: edge 0.831% vs pinnacle fair
  Oakland Athletics (Baltimore Orioles at Oakland Athletics) +131 at Caesars: edge 0.66% vs pinnacle fair
  Minnesota Twins (Chicago White Sox at Minnesota Twins) -114 at FanDuel: edge 0.657% vs pinnacle fair
  Minnesota Twins (Chicago White Sox at Minnesota Twins) -114 at DraftKings: edge 0.657% vs pinnacle fair


---

**More ParlayAPI resources**

- Docs: [parlay-api.com/docs](https://parlay-api.com/docs)
- Free API key (no card): [parlay-api.com/signup](https://parlay-api.com/signup)
- Browser calculators the math here matches: [no-vig](https://parlay-api.com/tools/no-vig-calculator), [parlay](https://parlay-api.com/tools/parlay-calculator), [EV](https://parlay-api.com/tools/ev-calculator)
- The rest of this series: [github.com/JacobiusMakes/parlayapi-notebooks](https://github.com/JacobiusMakes/parlayapi-notebooks)

These notebooks are for research and education. Nothing here is betting advice.